In [9]:
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
import numpy as np
from sklearn.model_selection import train_test_split
from code_files.data_preperation import prepare_for_train
from code_files.train import train
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

In [10]:
# Load Dataset
df_amazon = pd.read_csv("dataset/eda_amazon_sales_report.csv")
df_amazon.info()
df_amazon.columns
df_amazon.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117123 entries, 0 to 117122
Data columns (total 24 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Unnamed: 0                           117123 non-null  int64  
 1   Size                                 117123 non-null  int64  
 2   Qty                                  117123 non-null  int64  
 3   Amount                               117123 non-null  float64
 4   promotion-ids                        117123 non-null  int64  
 5   B2B                                  117123 non-null  int64  
 6   Status_Cancelled                     117123 non-null  bool   
 7   Status_Shipped                       117123 non-null  bool   
 8   Status_Shipped - Delivered to Buyer  117123 non-null  bool   
 9   Fulfilment_Amazon                    117123 non-null  bool   
 10  Fulfilment_Merchant                  117123 non-null  bool   
 11  ship-service-

,Unnamed: 0,Size,Qty,Amount,promotion-ids,B2B,Status_Cancelled,Status_Shipped,Status_Shipped - Delivered to Buyer,Fulfilment_Amazon,...,Category_Bottom,Category_Dupatta,Category_Ethnic Dress,Category_Saree,Category_Set,Category_Top,Category_Western Dress,Category_kurta,Month,Day
0,0,2,0,647.62,0,0,True,False,False,False,...,False,False,False,False,True,False,False,False,4,30
1,1,7,1,406.00,1,0,False,False,True,False,...,False,False,False,False,False,False,False,True,4,30
2,2,5,1,329.00,1,1,False,True,False,True,...,False,False,False,False,False,False,False,True,4,30
3,3,4,0,753.33,0,0,True,False,False,False,...,False,False,False,False,False,False,True,False,4,30
4,4,7,1,574.00,0,0,False,True,False,True,...,False,False,False,False,False,True,False,False,4,30


In [11]:
# split  data
dftrain, dfdev = train_test_split(df_amazon, test_size=0.1, random_state=42)
Xtrain, ytrain, Xdev, ydev = prepare_for_train(dftrain, dfdev)



In [12]:
# Grid Search for Random Forest
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt']
}

rf_model_grid = RandomForestRegressor(random_state=42)
rf_grid = GridSearchCV(
    estimator=rf_model_grid,
    param_grid=rf_param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2,
    scoring='neg_mean_absolute_percentage_error'
)

# Fit grid search
rf_grid.fit(Xtrain, ytrain)

print("\nBest Random Forest Parameters:", rf_grid.best_params_)
print("Best MAPE Score:", -rf_grid.best_score_)

Fitting 5 folds for each of 162 candidates, totalling 810 fits


/opt/anaconda3/envs/nbgrader-env/lib/python3.10/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/opt/anaconda3/envs/nbgrader-env/lib/python3.10/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this parameter as it is also the default value for RandomForestRegressors and ExtraTreesRegressors.
  warn(
/opt/anaconda3/envs/nbgrader-env/lib/python3.10/site-packages/sklearn/ensemble/_forest.py:416: FutureWarning: `max_features='auto'` has been deprecated in 1.1 and will be removed in 1.3. To keep the past behaviour, explicitly set `max_features=1.0` or remove this pa


Best Random Forest Parameters: {'max_depth': 10, 'max_features': 'auto', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best MAPE Score: 5.279017259286359e+16


In [15]:
# Get the best model from grid search
rf_model = rf_grid.best_estimator_

# Make predictions and calculate metrics using this best model
y_pred = rf_model.predict(Xdev)

# Calculate metrics
mae = mean_absolute_error(ydev, y_pred)
rmse = np.sqrt(mean_squared_error(ydev, y_pred))
r2 = r2_score(ydev, y_pred)

print("\nRandom Forest Results:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")


Random Forest Results:
MAE: 211.14
RMSE: 275.37
R2 Score: 0.05
[CV] END max_depth=5, max_features=auto, min_samples_leaf=1, min_samples_split=5, n_estimators=200; total time=  32.3s
[CV] END max_depth=5, max_features=auto, min_samples_leaf=2, min_samples_split=5, n_estimators=300; total time=  46.5s
[CV] END max_depth=5, max_features=auto, min_samples_leaf=4, min_samples_split=10, n_estimators=300; total time=  45.8s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=4, min_samples_split=2, n_estimators=100; total time=   6.4s
[CV] END max_depth=5, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=200; total time=  12.3s
[CV] END max_depth=8, max_features=auto, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=  21.1s
[CV] END max_depth=8, max_features=auto, min_samples_leaf=2, min_samples_split=2, n_estimators=100; total time=  20.7s
[CV] END max_depth=8, max_features=auto, min_samples_leaf=2, min_samples_split=5, n_estimators=100; to